# CRC-TissueNet: Colab / Kaggle quick start

Runs the whole project on a free GPU. **Runtime → Change runtime type → T4 GPU.**

Data and results are stored on Google Drive, so a disconnect never loses finished work. Re-running any cell skips finished runs and resumes unfinished ones.

| Phase | What | Approx. GPU time |
|---|---|---|
| 0 | setup + tests | 5 min |
| 1 | download data, reproduce MedMNIST v2 ResNet-18 | 1–2 h |
| 2 | CNN ablation → tuned base model | 2–4 h |
| 3 | transfer-learning strategies (E2) | 4–8 h |
| 4 | benchmark (E1 + E3) | 15–30 h, split across sessions and teammates |
| 5 | SHAP explanations | 10–30 min |

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Persist data and runs on Google Drive (skip this cell on Kaggle and set DRIVE = '/kaggle/working')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/crc-tissuenet'

In [ ]:
import os
REPO = 'https://github.com/bitsbyte111/pathmnist-crc-classification.git'   # <-- edit
if not os.path.exists('/content/crc-tissuenet'):
    !git clone -q $REPO /content/crc-tissuenet
%cd /content/crc-tissuenet
!git pull -q
!pip install -q -e ".[tune,dev]"

# data/, runs/ and reports/ live on Drive; the repo folder only holds symlinks
for d in ['data', 'runs', 'reports_out']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
for link, target in [('data', f'{DRIVE}/data'), ('runs', f'{DRIVE}/runs')]:
    if not os.path.islink(link):
        !rm -rf {link} && ln -s {target} {link}

In [ ]:
!pytest -q          # tests on synthetic data
!make smoke         # end-to-end pipeline check

## 1. Data + pipeline validation

The 224 px file is about 14 GB on disk. It is converted once into a memory-mapped array (RAM stays low) and kept on Drive.
If Drive space is tight, download only 28 px now and add 224 px before Phase 3.

In [ ]:
!python scripts/prepare_data.py --platform colab --sizes 28 224

In [ ]:
# Reproduce MedMNIST v2 ResNet-18 (28 px): published test accuracy 0.907, AUC 0.983
!python scripts/train.py --config configs/models/b2_resnet18.yaml -o eval.test=true
import json; m = json.load(open('runs/B2_resnet18/seed42/metrics.json'))
print('test acc', round(m['test']['accuracy'], 4), '| macro AUC', round(m['test']['macro_auc'], 4))

## 2. CNN ablation → tuned base model

18 stages: optimiser, learning rate, batch size, schedule, activation, initialisation, kernel size, width, depth, convs per block, pooling, head, batch norm, dropout, weight decay, augmentation, label smoothing and class weights.

See `docs/ABLATION_GUIDE.md` for the concept behind each stage.

In [ ]:
!python scripts/run_ablation.py --plan configs/ablation/cnn_ablation.yaml
!cp configs/models/b1_cnn_tuned.yaml $DRIVE/   # back up the tuned config

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('runs/cnn_ablation/ABLATION_REPORT.md').read()[:6000]))
display(Image('runs/cnn_ablation/figures/progress.png'))
display(Image('runs/cnn_ablation/figures/stage_activation.png'))

## 3. Transfer-learning strategies (validation only)

In [ ]:
!python scripts/run_suite.py --suite configs/suites/e2_transfer.yaml
!python scripts/report.py --suite configs/suites/e2_transfer.yaml --out $DRIVE/reports_out/e2_transfer

## 4. Benchmark on the independent test cohort (E1 + E3)

Run in several sessions or split between teammates with `--only`. Finished runs are skipped automatically.

In [ ]:
!python scripts/run_suite.py --suite configs/suites/e1_e3_benchmark.yaml --only B0_mlp B1_cnn B1_cnn_tuned B2_resnet18

In [ ]:
!python scripts/run_suite.py --suite configs/suites/e1_e3_benchmark.yaml --only P1_resnet50 P2_densenet121 P3_efficientnet_b0 P4_convnext_tiny

In [ ]:
!python scripts/run_suite.py --suite configs/suites/e1_e3_benchmark.yaml --only P5_deit_small P6_swin_tiny P7_mixer_b16

In [ ]:
!python scripts/report.py --suite configs/suites/e1_e3_benchmark.yaml --out $DRIVE/reports_out/e1_e3_benchmark
display(Markdown(open(f'{DRIVE}/reports_out/e1_e3_benchmark/REPORT.md').read()))

## 5. SHAP on the best model

In [ ]:
import pandas as pd
s = pd.read_csv(f'{DRIVE}/reports_out/e1_e3_benchmark/summary.csv')
best = s.sort_values('test_macro_f1_mean', ascending=False).id.iloc[0]
RUN = f'runs/e1_e3_benchmark/{best}/seed42'
print('Best model:', best)
!python scripts/explain.py --run $RUN --per-class 3
!python scripts/explain.py --run $RUN --per-class 3 --stain-space --no-sanity
# Also explain the tuned scratch CNN for comparison
!python scripts/explain.py --run runs/e1_e3_benchmark/B1_cnn_tuned/seed42 --per-class 3

In [ ]:
import glob, json
for f in sorted(glob.glob(f'{RUN}/explain/*.png')):
    print(f); display(Image(f, width=900))
print(json.load(open(f'{RUN}/explain/sanity_check.json'))['spearman_mean'])

## Optional: E4 colour augmentation, E5 resolution, Optuna

```
!python scripts/run_suite.py --suite configs/suites/e4_color_aug.yaml     # set the best model in the YAML first
!python scripts/run_suite.py --suite configs/suites/e5_resolution.yaml
!python scripts/tune.py --space configs/tuning/optuna_cnn.yaml
```